# UnderCover on AWS: proof of concept

This notebook proves the AWS route for the course one piece at a time: the UnderCover app on Lambda, then the UnderCover Agent on Bedrock AgentCore with Gateway, Policy and Memory.

It runs as the IAM user set up in the README. boto3 reads the same credentials as the AWS CLI, so there are no keys in this notebook. Everything it creates is named `undercover` and tagged `project=undercover`, which is how the last section finds it all and removes it.

## 1. Who we are running as

In [ ]:
import boto3
from botocore.exceptions import ClientError

session = boto3.Session()
identity = session.client("sts").get_caller_identity()
account_id = identity["Account"]

print(f"Account:  {account_id}")
print(f"Identity: {identity['Arn']}")
print(f"Region:   {session.region_name}")

## 2. A check of permissions

Each check is a read-only call to one of the services the course touches. If one fails, the message names the missing action, and the fix is to add it to `undercover-course-policy.json` and save the policy again in the console.

In [ ]:
def check(label, call):
    try:
        call()
        return label, "ok"
    except ClientError as error:
        details = error.response["Error"]
        return label, f"{details['Code']}: {details['Message']}"


agentcore = session.client("bedrock-agentcore-control")
project_tag = [{"Key": "project", "Values": ["undercover"]}]

checks = [
    check("AgentCore runtimes", lambda: agentcore.list_agent_runtimes()),
    check("AgentCore gateways", lambda: agentcore.list_gateways()),
    check("AgentCore memories", lambda: agentcore.list_memories()),
    check("AgentCore policy engines", lambda: agentcore.list_policy_engines()),
    check("CloudFormation stacks", lambda: session.client("cloudformation").describe_stacks()),
    check("Lambda functions", lambda: session.client("lambda").list_functions(MaxItems=1)),
    check("IAM roles", lambda: session.client("iam").list_roles(MaxItems=1)),
    check("S3 buckets", lambda: session.client("s3").list_buckets()),
    check("CloudWatch log groups", lambda: session.client("logs").describe_log_groups(limit=1)),
    check("Tag search", lambda: session.client("resourcegroupstaggingapi").get_resources(TagFilters=project_tag)),
    check("Budgets", lambda: session.client("budgets").describe_budgets(AccountId=account_id)),
]

for label, result in checks:
    print(f"{label:<28}{result}")

## 3. What this account has spent

The budget comes from AWS Budgets, which is free to query. If no budget exists, stop here and create one before deploying anything.

In [ ]:
budgets = session.client("budgets").describe_budgets(AccountId=account_id).get("Budgets", [])

if not budgets:
    print("No budget found. Create one in Billing and Cost Management, Budgets.")

for budget in budgets:
    spend = budget["CalculatedSpend"]
    limit = float(budget["BudgetLimit"]["Amount"])
    actual = float(spend["ActualSpend"]["Amount"])
    forecast = float(spend.get("ForecastedSpend", {}).get("Amount", 0))
    print(f"{budget['BudgetName']} ({budget['TimeUnit'].lower()})")
    print(f"  Spent so far: ${actual:,.2f} of ${limit:,.2f}")
    print(f"  Forecast:     ${forecast:,.2f}")

The breakdown by service comes from Cost Explorer, which charges one cent for each request. This cell makes a single request for the last 30 days and works out every figure from it. Credits and refunds are left out, so an account running on free credit still sees what its usage cost. Cost Explorer runs up to a day behind, so today's figure is usually incomplete.

In [ ]:
from collections import defaultdict
from datetime import date, timedelta

today = date.today()
request = {
    "TimePeriod": {
        "Start": (today - timedelta(days=30)).isoformat(),
        "End": (today + timedelta(days=1)).isoformat(),
    },
    "Granularity": "DAILY",
    "Metrics": ["UnblendedCost"],
    "GroupBy": [{"Type": "DIMENSION", "Key": "SERVICE"}],
    "Filter": {"Not": {"Dimensions": {"Key": "RECORD_TYPE", "Values": ["Credit", "Refund"]}}},
}

cost_explorer = session.client("ce")
by_day = defaultdict(float)
by_service = defaultdict(float)

while True:
    response = cost_explorer.get_cost_and_usage(**request)
    for day in response["ResultsByTime"]:
        for group in day["Groups"]:
            amount = float(group["Metrics"]["UnblendedCost"]["Amount"])
            by_day[day["TimePeriod"]["Start"]] += amount
            by_service[group["Keys"][0]] += amount
    if "NextPageToken" not in response:
        break
    request["NextPageToken"] = response["NextPageToken"]

month_start = today.replace(day=1).isoformat()
month_to_date = sum(amount for day, amount in by_day.items() if day >= month_start)

print(f"Today so far:  ${by_day.get(today.isoformat(), 0):,.2f}")
print(f"Month to date: ${month_to_date:,.2f}")
print(f"Last 30 days:  ${sum(by_day.values()):,.2f}")
print()
print("Last 30 days by service:")
for service, amount in sorted(by_service.items(), key=lambda item: -item[1]):
    if amount >= 0.005:
        print(f"  {service:<44}${amount:>8,.2f}")